# autoguidance — 00 · Dataset Builder  (RUN WITH INTERNET)

This notebook **prepares every offline artifact** the experiment runner needs.
It is the *only* notebook that touches the network. Run it on a Kaggle kernel with
**Internet = ON**, then point `10_experiment_runner.ipynb` (Internet = OFF) at the
Kaggle Datasets it uploads.

What it produces (one Kaggle Dataset per artifact):

| Artifact | Contents |
|---|---|
| LLaDA-8B weights | `GSAI-ML/LLaDA-8B-Instruct` snapshot incl. remote `modeling_*.py` |
| DiffusionGemma weights | `google/diffusiongemma-26B-A4B-it` snapshot |
| GPT-2 scorer | `gpt2-large` (Phase-1 generative perplexity) |
| MAUVE featurizer | the MAUVE GPT-2 feature model |
| Offline wheels (×2) | one wheelhouse per `transformers` pin — **tf5** (Gemma) and **tf446** (LLaDA) |
| HF data | `wikitext-103-v1` `save_to_disk` + `nltk punkt` |
| Code | the `autoguidance` package pinned at one commit |

**Pipeline:** auth → HF login → download weights → build BOTH wheel sets →
stage datasets → package code → tar everything → write `dataset-metadata.json` →
`kaggle datasets create`/`version` upload loop → verify + sha256 manifest.

> NOTE: the wheels MUST be built on the **same Kaggle base image** that the runner
> kernel uses, or `--no-index` install will fail on a binary mismatch. Pin
> `KAGGLE_BASE_IMAGE` below and use the identical image for both notebooks.


## 1 · CONFIG — every slug / id / path lives here

In [ ]:
# ----------------------------------------------------------------------------------
# SINGLE SOURCE OF TRUTH. Nothing below this cell should hardcode a slug or id.
# ----------------------------------------------------------------------------------
import os

# --- your Kaggle account ----------------------------------------------------------
KAGGLE_USERNAME = "aarush-dev"          # <-- the account that owns the datasets

# --- Kaggle Dataset slugs (one dataset per artifact) ------------------------------
# These MUST match KaggleConfig.slug_* defaults in autoguidance.config so the
# runner notebook can find them by the same name.
SLUGS = {
    "llada":          f"{KAGGLE_USERNAME}/autoguidance-llada-8b",
    "diffusiongemma": f"{KAGGLE_USERNAME}/autoguidance-diffusiongemma-26b",
    "gpt2_scorer":    f"{KAGGLE_USERNAME}/autoguidance-gpt2-large",
    "mauve":          f"{KAGGLE_USERNAME}/autoguidance-mauve-feat",
    "wheels":         f"{KAGGLE_USERNAME}/autoguidance-wheels",
    "hfdata":         f"{KAGGLE_USERNAME}/autoguidance-hfdata",
    "code":           f"{KAGGLE_USERNAME}/autoguidance-code",
}

# --- HuggingFace repo ids ---------------------------------------------------------
HF_IDS = {
    "llada":          "GSAI-ML/LLaDA-8B-Instruct",
    "diffusiongemma": "google/diffusiongemma-26B-A4B-it",
    "gpt2_scorer":    "gpt2-large",
}
# MAUVE uses a GPT-2 feature model under the hood; cache it as its own artifact.
MAUVE_FEATURIZER_ID = "gpt2-large"      # MAUVE default featurize_model_name

# --- code package -----------------------------------------------------------------
GIT_REPO_URL  = "https://github.com/aarush-dev/autoguidance.git"
PINNED_COMMIT = "REPLACE_WITH_FULL_40CHAR_SHA"   # <-- pin exactly; no moving HEAD

# --- base image: keep IDENTICAL between this notebook and the runner --------------
# Read it on Kaggle with:  !cat /etc/os-release ; pip --version ; python --version
KAGGLE_BASE_IMAGE = "gcr.io/kaggle-gpu-images/python:latest"   # record the digest you actually ran on

# --- transformers variants: we build TWO wheelhouses ------------------------------
# The runner picks one via BaseConfig.transformers_variant ("tf5" | "tf446").
TRANSFORMERS_VARIANTS = {
    "tf5":   "transformers>=5.12",   # DiffusionGemma needs new transformers
    "tf446": "transformers==4.46.3", # LLaDA custom modeling pinned to 4.46.x
}

# packages (besides transformers) that the runner installs OFFLINE. Keep in sync
# with autoguidance/requirements.txt. torch is preinstalled on the Kaggle image so
# we do NOT vendor it (it must match the image's CUDA build).
COMMON_REQUIREMENTS = [
    "accelerate", "safetensors", "sentencepiece", "datasets",
    "mauve-text", "scikit-learn", "scipy", "nltk", "sacrebleu",
    "einops", "tqdm", "pyyaml",
]

# --- local staging area on the internet kernel ------------------------------------
STAGE_DIR = "/kaggle/working/stage"      # everything is built here, then tar'd
os.makedirs(STAGE_DIR, exist_ok=True)

print("[config] KAGGLE_USERNAME =", KAGGLE_USERNAME)
print("[config] slugs:")
for k, v in SLUGS.items():
    print(f"          {k:16s} -> {v}")
print("[config] HF ids:", HF_IDS, "| MAUVE feat:", MAUVE_FEATURIZER_ID)
print("[config] transformers variants:", TRANSFORMERS_VARIANTS)
print("[config] pinned commit:", PINNED_COMMIT)
print("[config] base image    :", KAGGLE_BASE_IMAGE)
print("[config] STAGE_DIR      :", STAGE_DIR)


## 2 · Auth — Kaggle credentials + tooling

In [ ]:
# Write ~/.kaggle/kaggle.json from Kaggle Secrets (Add-ons -> Secrets).
# Secrets expected:  KAGGLE_USERNAME, KAGGLE_KEY, HF_TOKEN
import os, json, subprocess, sys
from kaggle_secrets import UserSecretsClient

sec = UserSecretsClient()
k_user = sec.get_secret("KAGGLE_USERNAME")
k_key  = sec.get_secret("KAGGLE_KEY")

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
kpath = os.path.expanduser("~/.kaggle/kaggle.json")
with open(kpath, "w") as f:
    json.dump({"username": k_user, "key": k_key}, f)
os.chmod(kpath, 0o600)
print("[auth] wrote", kpath, "for user", k_user)
assert k_user == KAGGLE_USERNAME, f"secret user {k_user!r} != CONFIG {KAGGLE_USERNAME!r}"

# tooling
print("[auth] installing kaggle + huggingface_hub ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "kaggle", "huggingface_hub"], check=True)

import kaggle, huggingface_hub
print("[auth] kaggle           :", kaggle.__version__)
print("[auth] huggingface_hub  :", huggingface_hub.__version__)


## 3 · HuggingFace login

In [ ]:
# Log in so gated repos (Gemma, LLaDA) resolve. Token comes from the HF_TOKEN secret.
from huggingface_hub import login, whoami
HF_TOKEN = sec.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
try:
    print("[hf] logged in as:", whoami(token=HF_TOKEN).get("name"))
except Exception as e:
    print("[hf] whoami failed (continuing):", e)


## 4 · Download model weights (incl. LLaDA remote *.py)

In [ ]:
# snapshot_download each repo into STAGE_DIR/<name>. allow_patterns=None -> everything,
# which is what we want for LLaDA so its custom modeling_*.py travels with the weights.
from huggingface_hub import snapshot_download

DL = {
    "llada":          (HF_IDS["llada"],          os.path.join(STAGE_DIR, "llada")),
    "diffusiongemma": (HF_IDS["diffusiongemma"], os.path.join(STAGE_DIR, "diffusiongemma")),
    "gpt2_scorer":    (HF_IDS["gpt2_scorer"],    os.path.join(STAGE_DIR, "gpt2_large")),
    "mauve":          (MAUVE_FEATURIZER_ID,      os.path.join(STAGE_DIR, "mauve_feat")),
}

for name, (repo_id, local_dir) in DL.items():
    os.makedirs(local_dir, exist_ok=True)
    print(f"\n[dl] {name}: {repo_id} -> {local_dir}")
    snapshot_download(
        repo_id=repo_id,
        local_dir=local_dir,
        local_dir_use_symlinks=False,   # real files, so tar captures bytes not links
        token=HF_TOKEN,
        # default ignore of nothing: pull *.py too (critical for LLaDA trust_remote_code)
    )
    # show what came down + size
    print(f"[dl] {name} contents:")
    for fn in sorted(os.listdir(local_dir))[:40]:
        print("       ", fn)

print("\n[dl] per-directory disk usage:")
for name, (_, local_dir) in DL.items():
    subprocess.run(["du", "-sh", local_dir])

# Sanity: LLaDA must ship its remote modeling file.
llada_dir = DL["llada"][1]
has_modeling = any(fn.startswith("modeling_") and fn.endswith(".py")
                   for fn in os.listdir(llada_dir))
print("[dl] LLaDA has remote modeling_*.py:", has_modeling)
assert has_modeling, "LLaDA snapshot missing modeling_*.py — runner trust_remote_code will fail"


## 5 · Build BOTH offline wheel sets (tf5 + tf446)

In [ ]:
# Two wheelhouses so the runner can install transformers>=5.12 (Gemma) OR
# transformers==4.46.3 (LLaDA) entirely offline. `pip download` resolves the full
# dependency closure for each pin. MUST run on the same base image as the runner.
WHEELS_ROOT = os.path.join(STAGE_DIR, "wheels")
os.makedirs(WHEELS_ROOT, exist_ok=True)

for variant, tf_spec in TRANSFORMERS_VARIANTS.items():
    wdir = os.path.join(WHEELS_ROOT, f"wheels_{variant}")
    os.makedirs(wdir, exist_ok=True)
    specs = [tf_spec] + COMMON_REQUIREMENTS
    print(f"\n[wheels] variant={variant} dir={wdir}")
    print(f"[wheels]   specs: {specs}")
    # NOTE: torch is intentionally excluded — it stays the image's preinstalled build.
    cmd = [sys.executable, "-m", "pip", "download", "-d", wdir] + specs
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print("[wheels] STDERR:\n", r.stderr[-3000:])
        raise RuntimeError(f"pip download failed for variant {variant}")
    n = len([x for x in os.listdir(wdir) if x.endswith((".whl", ".tar.gz"))])
    print(f"[wheels] variant={variant}: {n} files downloaded")
    subprocess.run(["du", "-sh", wdir])

print("\n[wheels] DONE. wheelhouse tree:")
subprocess.run(["du", "-sh", WHEELS_ROOT])


## 6 · Stage HF datasets (wikitext-103 + nltk punkt)

In [ ]:
# Materialize wikitext-103-v1 to disk and the punkt tokenizer so the runner needs
# zero network. Everything lands under STAGE_DIR/hfdata.
from datasets import load_dataset
import nltk

HFDATA_DIR = os.path.join(STAGE_DIR, "hfdata")
WIKITEXT_DIR = os.path.join(HFDATA_DIR, "wikitext-103-v1")
NLTK_DIR = os.path.join(HFDATA_DIR, "nltk_data")
os.makedirs(HFDATA_DIR, exist_ok=True)
os.makedirs(NLTK_DIR, exist_ok=True)

print("[data] loading wikitext-103-v1 ...")
ds = load_dataset("wikitext", "wikitext-103-v1")
print("[data] splits:", {k: len(v) for k, v in ds.items()})
ds.save_to_disk(WIKITEXT_DIR)
print("[data] saved wikitext ->", WIKITEXT_DIR)

print("[data] downloading nltk punkt + punkt_tab ->", NLTK_DIR)
nltk.download("punkt", download_dir=NLTK_DIR)
nltk.download("punkt_tab", download_dir=NLTK_DIR)

subprocess.run(["du", "-sh", HFDATA_DIR])
print("[data] hfdata staged.")


## 7 · Package the `autoguidance` code at the pinned commit

In [ ]:
# git archive the pinned commit into STAGE_DIR/code/autoguidance so the runner can
# `pip install -e`. Falls back to a shallow clone + checkout if archive is unavailable.
CODE_DIR = os.path.join(STAGE_DIR, "code")
PKG_DIR = os.path.join(CODE_DIR, "autoguidance")
os.makedirs(PKG_DIR, exist_ok=True)

work = "/kaggle/working/_clone"
subprocess.run(["rm", "-rf", work], check=True)
print(f"[code] cloning {GIT_REPO_URL} ...")
subprocess.run(["git", "clone", GIT_REPO_URL, work], check=True)
subprocess.run(["git", "-C", work, "checkout", PINNED_COMMIT], check=True)

# capture the exact commit we shipped
sha = subprocess.run(["git", "-C", work, "rev-parse", "HEAD"],
                     capture_output=True, text=True).stdout.strip()
print("[code] shipping commit:", sha)
assert PINNED_COMMIT == "REPLACE_WITH_FULL_40CHAR_SHA" or sha == PINNED_COMMIT, \
    "checked-out commit does not match PINNED_COMMIT"

# export a clean tree (no .git) into PKG_DIR
subprocess.run(f"git -C {work} archive {sha} | tar -x -C {PKG_DIR}", shell=True, check=True)
# record provenance
with open(os.path.join(PKG_DIR, "SHIPPED_COMMIT.txt"), "w") as f:
    f.write(sha + "\n")

print("[code] package tree:")
for fn in sorted(os.listdir(PKG_DIR)):
    print("       ", fn)
assert os.path.exists(os.path.join(PKG_DIR, "setup.py")), "setup.py missing — pip install -e will fail"
subprocess.run(["du", "-sh", CODE_DIR])


## 8 · Tar one `.tar.gz` per artifact

In [ ]:
# Tar each staged dir so uploads are a single fat file (Kaggle handles big tars well,
# and the runner extracts only the model it needs). Loop over artifact -> source dir.
import tarfile, time

UPLOAD_ROOT = "/kaggle/working/upload"   # one subfolder per dataset
ARTIFACTS = {
    "llada":          os.path.join(STAGE_DIR, "llada"),
    "diffusiongemma": os.path.join(STAGE_DIR, "diffusiongemma"),
    "gpt2_scorer":    os.path.join(STAGE_DIR, "gpt2_large"),
    "mauve":          os.path.join(STAGE_DIR, "mauve_feat"),
    "wheels":         os.path.join(STAGE_DIR, "wheels"),
    "hfdata":         os.path.join(STAGE_DIR, "hfdata"),
    "code":           os.path.join(STAGE_DIR, "code"),
}

def tar_dir(src, dst_tgz):
    t0 = time.time()
    base = os.path.basename(src.rstrip("/"))
    with tarfile.open(dst_tgz, "w:gz") as tar:
        tar.add(src, arcname=base)
    print(f"[tar] {dst_tgz}  ({os.path.getsize(dst_tgz)/1e9:.2f} GB, {time.time()-t0:.0f}s)")

for name, src in ARTIFACTS.items():
    folder = os.path.join(UPLOAD_ROOT, name)
    os.makedirs(folder, exist_ok=True)
    tar_dir(src, os.path.join(folder, f"{name}.tar.gz"))

print("\n[tar] upload tree:")
subprocess.run(["du", "-sh", UPLOAD_ROOT])


## 9 · Write `dataset-metadata.json` per folder

In [ ]:
# Each upload folder needs a dataset-metadata.json with a fully-qualified id.
import json

TITLES = {
    "llada":          "autoguidance LLaDA-8B-Instruct weights",
    "diffusiongemma": "autoguidance DiffusionGemma-26B-A4B-it weights",
    "gpt2_scorer":    "autoguidance gpt2-large scorer",
    "mauve":          "autoguidance MAUVE featurizer",
    "wheels":         "autoguidance offline wheels (tf5 + tf446)",
    "hfdata":         "autoguidance HF data (wikitext-103 + punkt)",
    "code":           "autoguidance code package",
}

for name in ARTIFACTS:
    folder = os.path.join(UPLOAD_ROOT, name)
    meta = {
        "title": TITLES[name],
        "id": SLUGS[name],                          # <user>/<slug>
        "licenses": [{"name": "CC0-1.0"}],
    }
    mpath = os.path.join(folder, "dataset-metadata.json")
    with open(mpath, "w") as f:
        json.dump(meta, f, indent=2)
    print(f"[meta] {mpath}: id={meta['id']}")


## 10 · Upload loop (create, else version)

In [ ]:
# First publish: `kaggle datasets create`. If the dataset already exists, fall back
# to `kaggle datasets version`. -u uploads files individually (handles big tars).
import subprocess, datetime

def run(cmd):
    print("[upload] $", " ".join(cmd))
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout)
    if r.stderr:
        print("[upload] stderr:", r.stderr)
    return r.returncode, (r.stdout + r.stderr)

stamp = datetime.datetime.utcnow().isoformat(timespec="seconds")
for name in ARTIFACTS:
    folder = os.path.join(UPLOAD_ROOT, name)
    print(f"\n===== {name} ({SLUGS[name]}) =====")
    rc, out = run(["kaggle", "datasets", "create", "-p", folder, "-u", "--dir-mode", "tar"])
    if rc != 0 and ("already exists" in out or "409" in out or "Conflict" in out):
        print("[upload] exists -> versioning")
        run(["kaggle", "datasets", "version", "-p", folder,
             "-m", f"update {stamp}", "--dir-mode", "tar"])
    print(f"[upload] {name} -> https://www.kaggle.com/datasets/{SLUGS[name]}")


## 11 · Verify + sha256 manifest

In [ ]:
# Confirm the datasets are visible to your account, then print a size + sha256 table
# of everything we tarred (so the runner can spot a corrupt download).
import hashlib

print("[verify] kaggle datasets list -m (mine):")
subprocess.run(["kaggle", "datasets", "list", "-m"])

def sha256(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(buf), b""):
            h.update(chunk)
    return h.hexdigest()

print(f"\n{'artifact':16s} {'size(GB)':>9s}  sha256")
for name in ARTIFACTS:
    tgz = os.path.join(UPLOAD_ROOT, name, f"{name}.tar.gz")
    if os.path.exists(tgz):
        print(f"{name:16s} {os.path.getsize(tgz)/1e9:9.2f}  {sha256(tgz)}")
print("\n[verify] DONE. Attach these datasets to 10_experiment_runner (Internet OFF).")
